In [13]:
# CARGA DE DATOS #

import pandas as pd
# test comment
# Ruta al archivo de Excel
archivo_excel = './IIYNT-REQ-BD-003.xlsx' #../

# Número de la fila desde la cual querés empezar (0-indexado)
fila_inicio = 3

# Leer el archivo Excel
init_df = pd.read_excel(archivo_excel)

# Mostrar las primeras filas del DataFrame para confirmar que se leyó bien
init_df.head()

# TRANSFORMACIÓN DE DATOS #

desired_columns = [
    'Temperatura Ambiental (°C)', 
    'Humedad',
    'Temperatura de la muestra (°C)', 
    'pH', 
    'CE\n(µS/cm)', 
    'STD\n(mg/L)',
    'STS\n(mL sed/L)', 
    'OD\n(mg/L)', 
    'Nivel (cm)', 
    'Turbidez (NTU)',
    'Dureza\n(mg CaCO3/L)', 
    'Cloruros\n(mg Cl-/L)'
]
df = init_df.iloc[2:123][desired_columns]
df


,Temperatura Ambiental (°C),Humedad,Temperatura de la muestra (°C),pH,CE\n(µS/cm),STD\n(mg/L),STS\n(mL sed/L),OD\n(mg/L),Nivel (cm),Turbidez (NTU),Dureza\n(mg CaCO3/L),Cloruros\n(mg Cl-/L)
2,16,0.929,18,8.2,1240,610,NaN,5.37,NaN,NaN,NaN,NaN
3,17,0.47,19,8.3,1630,810,1.8,4.3,NaN,NaN,147,156
4,11.9,0.47,13,8.1,1000,490,18,5.3,NaN,41.2,94,78
5,11.9,0.47,13,8.2,1000,490,18,4.67,NaN,38.9,86,82
6,11.9,0.47,13,8.3,1350,670,0.1,7.01,NaN,30.7,200,117
...,...,...,...,...,...,...,...,...,...,...,...,...
118,12.8,0.57,15.3,8,1110,550,18,4.18,25,5.04,316,66
119,12.8,0.57,15.3,8.1,1080,550,18,4.23,25,5.69,316,66
120,12.8,0.57,15.4,8.4,1050,520,44,1.96,70,113,146,82
121,12.8,0.57,15.4,8.4,1070,530,44,1.67,70,133,146,82


In [14]:

#ver los que tienen valores nulos o no asignados

#df.isnull().sum()
df.isna().sum()

#Eliminar las filas que tienen valores nulos o no asignados
#df.dropna(inplace=True) #aca se modifica el Dataframe original con inplace=true

#Si asigno a nueva variable no hace falta inplace=true pero no modifica el Dataframe original
#df_sin_nulos=df.dropna()

#Como opcion puedo llenar los valores nulos con datos adecuados: 
#fillna(method='bfill') (llenar con el valor siguiente de la columna)
#fillna(method='ffill') (llenar con el valor anterior de la columna)
#fillna(0) (llenar con ceros) 

mediaNivel=df['Nivel (cm)'].mean()
df['Nivel (cm)']=df['Nivel (cm)'].fillna(mediaNivel)

mediaSTS=df['STS\n(mL sed/L)'].mean()
df['STS\n(mL sed/L)']=df['STS\n(mL sed/L)'].fillna(mediaSTS)

mediaTurbidez=df['Turbidez (NTU)'].mean()
df['Turbidez (NTU)']=df['Turbidez (NTU)'].fillna(mediaTurbidez)

mediaDureza=df['Dureza\n(mg CaCO3/L)'].mean()
df['Dureza\n(mg CaCO3/L)']=df['Dureza\n(mg CaCO3/L)'].fillna(mediaDureza)

mediaCloruros=df['Cloruros\n(mg Cl-/L)'].mean()
df['Cloruros\n(mg Cl-/L)']=df['Cloruros\n(mg Cl-/L)'].fillna(mediaCloruros)

df_sin_nulos=df.copy()
df_sin_nulos.isna().sum()

#Eliminar las filas que tienen valores nulos o no asignados
df_sin_nulos.dropna(inplace=True) #aca se modifica el Dataframe original con inplace=true
df_sin_nulos.isna().sum()

Temperatura Ambiental (°C)        0
Humedad                           0
Temperatura de la muestra (°C)    0
pH                                0
CE\n(µS/cm)                       0
STD\n(mg/L)                       0
STS\n(mL sed/L)                   0
OD\n(mg/L)                        0
Nivel (cm)                        0
Turbidez (NTU)                    0
Dureza\n(mg CaCO3/L)              0
Cloruros\n(mg Cl-/L)              0
dtype: int64

In [15]:
def min_max_info(dataframe: pd.DataFrame) -> pd.DataFrame:
    name = list()
    minimums = list()
    maximums = list()
    for i, key in enumerate(dataframe.columns):
        name.append(key)
        minimums.append(dataframe[key].min())
        maximums.append(dataframe[key].max())
    return pd.DataFrame(
        {
            'name': name,
            'mins': minimums,
            'maxs': maximums
        }
    )
min_max_info(df_sin_nulos)

,name,mins,maxs
0,Temperatura Ambiental (°C),10.40,26.000
1,Humedad,0.36,0.929
2,Temperatura de la muestra (°C),12.80,28.100
3,pH,7.20,8.700
4,CE\n(µS/cm),200.00,1690.000
5,STD\n(mg/L),140.00,840.000
6,STS\n(mL sed/L),0.10,650.000
7,OD\n(mg/L),0.00,9.120
8,Nivel (cm),20.00,70.000
9,Turbidez (NTU),2.68,1000.000


In [16]:
def normalize(dataframe: pd.DataFrame, min: float, max: float) -> pd.DataFrame:
    return (dataframe - min) / (max - min)

def denormalize(dataframe: pd.DataFrame, min: float, max: float) -> pd.DataFrame:
    return dataframe * (max - min) + min


def normalizer(dataframe: pd.DataFrame) -> pd.DataFrame:
    df = dataframe.copy()
    df['Temperatura Ambiental (°C)']                = normalize(df['Temperatura Ambiental (°C)'],       min=0, max=40)
    df['Humedad'] = df['Humedad']
    df['Temperatura de la muestra (°C)']            = normalize(df['Temperatura de la muestra (°C)'],   min=0, max=40)
    df['pH']                                        = normalize(df['pH'],                               min=6, max=10)
    df['CE\n(µS/cm)']                               = normalize(df['CE\n(µS/cm)'],                      min=100, max=2000)
    df['STD\n(mg/L)']                               = normalize(df['STD\n(mg/L)'],                      min=100, max=1000)
    df['STS\n(mL sed/L)']                           = normalize(df['STS\n(mL sed/L)'],                  min=0, max=1000)
    df['OD\n(mg/L)']                                = normalize(df['OD\n(mg/L)'],                       min=0, max=10)
    df['Nivel (cm)']                                = normalize(df['Nivel (cm)'],                       min=0, max=100)
    df['Turbidez (NTU)']                            = normalize(df['Turbidez (NTU)'],                   min=0, max=1000)
    df['Dureza\n(mg CaCO3/L)']                      = normalize(df['Dureza\n(mg CaCO3/L)'],             min=50, max=500)
    df['Cloruros\n(mg Cl-/L)']                      = normalize(df['Cloruros\n(mg Cl-/L)'],             min=0, max=200)    
    return df

def denormalizer(dataframe: pd.DataFrame) -> pd.DataFrame:
    df = dataframe.copy()
    df['Temperatura Ambiental (°C)']                = denormalize(df['Temperatura Ambiental (°C)'],       min=0, max=40)
    df['Humedad'] = df['Humedad']
    df['Temperatura de la muestra (°C)']            = denormalize(df['Temperatura de la muestra (°C)'],   min=0, max=40)
    df['pH']                                        = denormalize(df['pH'],                               min=6, max=10)
    df['CE\n(µS/cm)']                               = denormalize(df['CE\n(µS/cm)'],                      min=100, max=2000)
    df['STD\n(mg/L)']                               = denormalize(df['STD\n(mg/L)'],                      min=100, max=1000)
    df['STS\n(mL sed/L)']                           = denormalize(df['STS\n(mL sed/L)'],                  min=0, max=1000)
    df['OD\n(mg/L)']                                = denormalize(df['OD\n(mg/L)'],                       min=0, max=10)
    df['Nivel (cm)']                                = denormalize(df['Nivel (cm)'],                       min=0, max=100)
    df['Turbidez (NTU)']                            = denormalize(df['Turbidez (NTU)'],                   min=0, max=1000)
    df['Dureza\n(mg CaCO3/L)']                      = denormalize(df['Dureza\n(mg CaCO3/L)'],             min=50, max=500)
    df['Cloruros\n(mg Cl-/L)']                      = denormalize(df['Cloruros\n(mg Cl-/L)'],             min=0, max=200)    
    return df




In [17]:
from keras.layers import Input, Dense # tipos de capas de la red neuronal:
                                      # Dense = Conectado todo con todo
                                      # Input = Solo entrada
from keras.activations import sigmoid # Función de activación a la salida de la neurona
from keras.losses import mse # Tipo de ajuste de error (mean square error)
from keras.models import Model # Clase que devuelve la instancia del modelo
                               # el modelo es de la forma entrada -> salida
import numpy as np
import os as os

from tensorflow.keras.callbacks import TensorBoard # Interfaz Web de Tensorflow para visualizar
                                                   # grafico de entrenamiento
import shutil # Utilidad para manejo de consola
try:
    shutil.rmtree('./.logs') # Si existe, borro la carpeta .logs que es necesaria
                             # para la ingesta de tensorboard
except FileNotFoundError:
    pass
os.mkdir('./.logs') # creo una carpeta vacia
                    # para entrenar, ejecutar en consola y dentro del entorno virtrual de python
                    # lo siguiente: tensorboard --logdir=./.logs
vector_size = len(df_sin_nulos.columns) # defino tamaño de vector de datos (en este caso 12 columnas)
dataset = np.array( # Lo hago matriz, papu
    normalizer(df_sin_nulos).astype(np.float16) # Normalizo el dataset inicial sin nulos y lo convierto a float de 16bit
)
encoded_size = 3 # Me invento un tamaño para el vector clasificador

# definimos modelo: (notación algebraica simulada: x = f(y))
# recomiendo que le preguntes a chatGPT el tema de los shapes de keras porque es un quilombo
input_layer = Input(shape=(vector_size, )) # buscar en chatgpt maneras de construir una red neuronal con keras

x = Dense(4, activation=sigmoid)(input_layer) # se colocan unicamente 4 neuronas porque se me canta
# nota: Instancio la clase y la llamo, esto es un diseño "raro" de keras para que la definición
# de la red se "parezca" a una notacion matematica

encoded = Dense(encoded_size, activation=sigmoid)(x) #seria como la matriz de pesos por el vector de entrada x
x = Dense(4, activation=sigmoid)(encoded)
output = Dense(vector_size, activation=sigmoid)(x) # metodo call

autoencoder = Model(input_layer, output) # creo un modelo a partir de lo definido arriba
autoencoder.compile(optimizer='adam', loss='mse') # se compila el modelo
autoencoder.summary() # se imprime un resumen


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 12)]              0         
                                                                 
 dense (Dense)               (None, 4)                 52        
                                                                 
 dense_1 (Dense)             (None, 3)                 15        
                                                                 
 dense_2 (Dense)             (None, 4)                 16        
                                                                 
 dense_3 (Dense)             (None, 12)                60        
                                                                 
Total params: 143 (572.00 Byte)
Trainable params: 143 (572.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [18]:

tensorboard = TensorBoard(log_dir='./.logs') #creo instancia para guardar
epochs = 12000 # cantidad de veces a entranar o pasos para realizar el descenso del gradiente
autoencoder.fit(
    x=dataset,
    y=dataset,
    epochs=epochs,
    batch_size=len(dataset), #tamaño del bloque de entrenamiento
    shuffle=True,
    verbose=0, # que no envie salidas
    callbacks=[tensorboard]
    
)
    


In [19]:
generator = Model(encoded, output) #va del vector de clasificacion hasta la salida
generator.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 3)]               0         
                                                                 
 dense_2 (Dense)             (None, 4)                 16        
                                                                 
 dense_3 (Dense)             (None, 12)                60        
                                                                 
Total params: 76 (304.00 Byte)
Trainable params: 76 (304.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [20]:
encoder = Model(input_layer, encoded) #va desde el input a los datos codificados o clasificador
encoder.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 12)]              0         
                                                                 
 dense (Dense)               (None, 4)                 52        
                                                                 
 dense_1 (Dense)             (None, 3)                 15        
                                                                 
Total params: 67 (268.00 Byte)
Trainable params: 67 (268.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [21]:
amount = 500 #cantidad de predicciones
generated = np.zeros(shape=(amount, vector_size))
params = np.random.rand(amount, encoded_size)
generated = generator.predict(params)
gen_df = pd.DataFrame(generated)
gen_df.columns = df_sin_nulos.columns
gen_df = denormalizer(gen_df)
gen_df.to_csv('./dataset_simulado_nata.csv')


16/16 [==============================] - 0s 961us/step


In [22]:
min_max_info(df_sin_nulos)

,name,mins,maxs
0,Temperatura Ambiental (°C),10.40,26.000
1,Humedad,0.36,0.929
2,Temperatura de la muestra (°C),12.80,28.100
3,pH,7.20,8.700
4,CE\n(µS/cm),200.00,1690.000
5,STD\n(mg/L),140.00,840.000
6,STS\n(mL sed/L),0.10,650.000
7,OD\n(mg/L),0.00,9.120
8,Nivel (cm),20.00,70.000
9,Turbidez (NTU),2.68,1000.000


In [23]:
min_max_info(gen_df)

,name,mins,maxs
0,Temperatura Ambiental (°C),11.487352,26.133339
1,Humedad,0.244158,0.933558
2,Temperatura de la muestra (°C),13.999791,20.548275
3,pH,7.278951,8.462933
4,CE\n(µS/cm),335.199310,1945.801025
5,STD\n(mg/L),191.090637,973.825134
6,STS\n(mL sed/L),10.436943,706.751953
7,OD\n(mg/L),0.122127,8.867413
8,Nivel (cm),14.195447,62.054626
9,Turbidez (NTU),3.202071,999.196167


In [24]:
colors = encoder.predict(dataset)
colored_dataset = df_sin_nulos.copy() #copio el dataset original y lo coloreo con el clsificador (encoder)



def color_row(row, color_vector):
    color_rgb = tuple((color_vector*255).astype(np.int16))
    color_hex = '#{:02X}{:02X}{:02X}'.format(color_rgb[0], color_rgb[1], color_rgb[2])
    return [f'background-color: {color_hex};' for _ in row.index]

# Create an iterator for the colors
color_iterator = iter(colors)

# Apply colors to each row
styled_df = colored_dataset.style.apply(lambda row: color_row(row, next(color_iterator)), axis=1)
styled_df.to_excel("colored_dataset.xlsx", engine='openpyxl')


4/4 [==============================] - 0s 2ms/step
